# Fig. 1: Density Structure of Halo 2

Spherically averaged density of every DM particle at z = 0 in 80 logarithmic shells, with
the NFW and Einasto fits, and the projected density along z in a 1000 kpc box.

Needs `$AURIGA_DATA`. Writes `fig1_*.dat` and `fig1_meta.gnu` to `output/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # AurigaDFs/, so the package imports

import h5py
import numpy as np

from Empirical_DF_Construction.core.density_fits import einasto_shape, fit_nfw_einasto_gridsearch, nfw_shape
from Empirical_DF_Construction.io.auriga_root import resolve_auriga_data_root
from Empirical_DF_Construction.io.centering import load_center_from_catalog
from Empirical_DF_Construction.io.files import groupcat_files, snapshot_files
from Empirical_DF_Construction.io.header import read_header
from Empirical_DF_Construction.io.layout import resolve_data_dir_for_snap, resolve_halo_dir
from Empirical_DF_Construction.io.virial import load_R200m_kpc
from Empirical_DF_Construction.paper_figures.common import REF_HALO, SNAP, out_dir

EDGES = np.logspace(np.log10(0.2), np.log10(1e4), 81)   # kpc, 80 shells
SOFTENING_KPC = 0.369
HALF_BOX_KPC = 500.0
NPIX = 500
CHUNK = 2_000_000
out = out_dir()

## Particles: shell counts and the projected map in one pass

In [ ]:
data_dir = resolve_data_dir_for_snap(resolve_halo_dir(resolve_auriga_data_root(), REF_HALO), SNAP)
sfiles, gfiles = snapshot_files(data_dir, SNAP), groupcat_files(data_dir, SNAP)
hdr = read_header(sfiles[0])
centre, _, _ = load_center_from_catalog(groupcat_files_list=gfiles, center_mode="subhalo", a=hdr.a)
to_kpc = 1000.0 / hdr.h * hdr.a
mpart = hdr.mass_table[1] * 1e10 / hdr.h

counts = np.zeros(len(EDGES) - 1)
pix = np.linspace(-HALF_BOX_KPC, HALF_BOX_KPC, NPIX + 1)
image = np.zeros((NPIX, NPIX))
for fn in sfiles:
    with h5py.File(fn, "r") as f:
        pos = f["PartType1/Coordinates"]
        for start in range(0, pos.shape[0], CHUNK):
            x = (pos[start:start + CHUNK] - centre) * to_kpc
            counts += np.histogram(np.linalg.norm(x, axis=1), bins=EDGES)[0]
            image += np.histogram2d(x[:, 1], x[:, 0], bins=[pix, pix])[0]
r200m = load_R200m_kpc(gfiles=gfiles, a=hdr.a, h=hdr.h)

## Radial profile and fits (2-300 kpc, shells with at least 200 particles)

In [ ]:
r = np.sqrt(EDGES[1:] * EDGES[:-1])
rho = counts * mpart / (4 / 3 * np.pi * (EDGES[1:] ** 3 - EDGES[:-1] ** 3))
k = counts > 0
r, rho, counts = r[k], rho[k], counts[k]
fit = fit_nfw_einasto_gridsearch(r, rho, counts, r_fit_min=2.0, r_fit_max=300.0, min_counts=200)
nfw, ein = fit["nfw"], fit["einasto"]
print("NFW", nfw)
print("Einasto", ein)
print(f"R200m = {r200m:.1f} kpc")

## Export

In [ ]:
np.savetxt(out / "fig1_profile.dat", np.column_stack([r, rho]), header="r_kpc rho")
np.savetxt(out / "fig1_nfw.dat", np.column_stack([r, nfw["rhos"] * nfw_shape(r, nfw["rs"])]))
np.savetxt(out / "fig1_einasto.dat",
           np.column_stack([r, ein["rho2"] * einasto_shape(r, ein["r2"], ein["alpha"])]))

sigma = image * mpart / (pix[1] - pix[0]) ** 2
logsig = np.log10(np.where(sigma > 0, sigma, np.nan))
lo, hi = np.nanpercentile(logsig, [2, 98])
c = 0.5 * (pix[1:] + pix[:-1])
with open(out / "fig1_map.dat", "w") as fh:
    fh.write("# x_kpc y_kpc log10_sigma\n")
    for j, y in enumerate(c):
        np.savetxt(fh, np.column_stack([c, np.full(NPIX, y), np.nan_to_num(logsig[j], nan=lo)]), fmt="%.6g")
        fh.write("\n")

meta = [f"SOFTENING = {SOFTENING_KPC}", f"R200M = {r200m:.6g}", f"HALF_BOX = {HALF_BOX_KPC:g}",
        f"SIGMA_MIN = {lo:.4g}", f"SIGMA_MAX = {hi:.4g}"]
(out / "fig1_meta.gnu").write_text("\n".join(meta) + "\n")